# Recherche directe de politique

Dans ce TP, l'objectif est d'implémenter un algorithme de recherche directe de politique pour trouver une politique optimale pour différents environnements.

> Vous devez  <span style="color:red"> commenter votre code ! </span> 
> 
> Vous devez aussi répondre à des questions (dans les cellules <span style="color:blue">Votre réponse: </span> )


> Le code doit être fonctionnel avec l'environnement virtuel du TP. Si d'autres packages que ceux présents dans l'environnement virtuel créé au départ sont nécessaires, vous devez ajouter à votre dépôt un fichier `environnement.yaml` qui est un export de votre environnement virtuel. Ce fichier est obtenu avec la commande suivante:  ```conda env export > environnement.yaml```

In [4]:
import gymnasium as gym
import random
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
import torch
from torch.distributions import Categorical
%matplotlib inline
%load_ext autoreload

def init_seed(seedval):
    torch.manual_seed(seedval)
    np.random.seed(seedval)
    random.seed(seedval)

# 1 Environnement CartPole

## 1.1 Découverte environnement

Lire la documentation de l'environnement [CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/).

Bien comprendre les caractéristiques de cet environnement et compléter ci-dessous:

- états/actions discrets ou continus ? 
- dimensions des espaces d'états et d'actions ?
- signification de *terminated* et *truncated* ?
- durée max d'un épisode ?
- somme des récompenses max par épisode ?

> <span style="color:blue">Votre réponse: </span>


## 1.2 Définition de la Politique


> <span style="color:green">- Quelle est la différence entre une politique stochastique et déterministe ?  </span>



> <span style="color:blue">Votre réponse: </span>

> <span style="color:green">- Dans le cas d'une politique déterministe dans le CartPole, qu'est-ce qui sera en entrée de la politique ? En sortie ?  </span>

> <span style="color:blue">Votre réponse: </span>

> <span style="color:green">- Dans le cas d'une politique stochastique dans le CartPole, qu'est-ce qui sera en entrée de la politique ? En sortie ?  </span>

> <span style="color:blue">Votre réponse: </span>

Compléter la classe *Politique* ci-dessous qui sera utilisée dans le CartPole. Cette politique sera déterministe ou stochastique (choix via l'argument *det*) et sera composée d'une seule couche de poids. 
- Vous devez ici utiliser *numpy* et *ndarray* pour les poids, et pas les couches linéaires de *Torch* car vous n'aurez pas besoin ici de calcul de gradient. 
- Pour la politique stochastique, vous utiliserez une distribution catégorielle de *Torch* 

In [2]:

class Politique():

    def __init__(self, dim_entree:int, dim_sortie:int, det = True):
        
        #TO DO 

        
 
    def output(self, etat : np.ndarray) -> int:

        #TO DO
 
 
    def set_poids(self, poids: np.ndarray):
       
    
    def get_poids(self) -> np.ndarray:
        

    def save(self,file):
        

    def load(self,file):
        

Compléter la méthode *eval_politique* qui évalue une politique sur plusieurs épisodes, et renvoie la **performance de chaque épisode**, qui est la somme des récompenses obtenues sur chaque épisode.

NB: L'attribut 'init_large' permet d'élargir les bornes définies pour le choix de l'état initial.

In [ ]:
def eval_politique(politique, env, seed=5, nb_episodes=100, max_t=1000, same_init=False, init_large=False) -> list:
    """

    Args:
        politique: politique à évaluer
        env: environnement sur lequel on évalue la politique
        seed: graine pour initialiser l'environnement (et donc la séquence d'états initiaux); si
        same_init: si True, on prendra toujours le même état initial sur les episodes (celui défini par la seed); 
                   si False on prendra differents etats initiaux pour chaque épisode (la seed fixe ici la séquence)
        nb_episodes: nombre d'épisodes sur lesquels on évalue la politique
        max_t : nombre maximum de pas d'un épisode
        init_large: si True, élargit les bornes définies pour le choix de l'état initial.

    Returns:
        list:  somme des récompenses obtenues sur chaque épisode
    """
    env.reset(seed=seed) #    init la sequence d etats initiaux: les appels a env.reset() ensuite donneront toujours la meme séquence
    somme_rec_per_epi = []
    for epi in range(0, nb_episodes):
        recs = []
        if init_large:
            if same_init:
                state, _ = env.reset(seed=seed,options={'low':-0.2,'high':0.2})#
            else:
                state, _ = env.reset(options={'low':-0.2,'high':0.2})
        else:
            if same_init:
                state, _ = env.reset(seed=seed)
            else:
                state, _ = env.reset()
        
        for t in range(max_t):
            #TO COMPLETE
            
        
        
    return somme_rec_per_epi


On évalue ci-dessous une politique déterministe et une politique stochastique (non-optimisées) sur 5 épisodes, **à partir du même état initial** pour chaque épisode.

In [ ]:
env = gym.make('CartPole-v1',render_mode="human")
init_seed(5)

politique = Politique(dim_entree=4, dim_sortie=2,det=True) 
perf = eval_politique(politique,env,seed=5,nb_episodes=5,same_init=True)
print(f"Perf  = {perf}")

politique = Politique(dim_entree=4, dim_sortie=2,det=False) 
perf = eval_politique(politique,env,seed=5,nb_episodes=5,same_init=True)
print(f"Perf  = {perf}")

> <span style="color:green">- Analyser les résultats obtenus sur les 2 évaluations: quelle est la différence ? pourquoi ? que peut-on en déduire sur la stochasticité de l'environnement Cart-Pole ? Expliquer/justifier.  </span>


> <span style="color:blue">Votre réponse: </span>

## 1.3 Algorithme de recherche directe de politique

L'objectif est d'implément un algorithme de type *Adaptive Noise Scaling*, qui va à chaque itération, générer une nouvelle politique candidate en perturbant les paramètres de la meilleure politique trouvée. 

La perturbation des paramètres sera réalisée en ajoutant un *bruit* sur chaque paramètre, et en faisant **varier la variance** de ce bruit: variance réduite si la politique candidate trouvée est meilleure que l'ancienne, augmentée sinon.

L'algorithme est le suivant:
- Choix d'une politique initiale arbitraire Pi et évaluation de cette politique
- Tant que l'environnement n'est pas résolu 
    - Génération d'une nouvelle politique candidate à partir de Pi
    - Evaluation de la candidate
    - Si la candidate a une meilleure perf que Pi: réduction de la variance du bruit et Pi = Candidate
    - Si la candidate a une moins bonne perf que Pi: augmentation de la variance du bruit 

Pour perturber un poids: `w = w + bruit_std * np.random.randn(1)` et pour réduire la variance du bruit: `bruit_std = max(1e-3, bruit_std / 2)`.

> <span style="color:green">Vous devez  </span>
- compléter la méthode *recherche_directe* pour faire une recherche dans l'espace des paramètres de la politique. Cette méthode renverra les performances obtenues à chaque itération et les poids de la meilleure politique trouvée



In [ ]:
def recherche_directe(politique, env,  nb_episodes=5000, max_t=1000) -> tuple[list,np.ndarray] :
    """_summary_

    Args:
        politique: politique à optimiser
        env: environnement sur lequel on évalue la politique
        nb_episodes : nb d'episodes max pour la recherche 
        max_t : nb max de pas par épisode 

    Returns:
        tuple[list,np.ndarray]: liste des performances à chaque épisode de la recherche, et meilleur_poids trouvé
    """
    #TO COMPLETE
    bruit_std = 1e-2
    
    for i_episode in range(1, nb_episodes+1):
         
        if perf >= meilleur_perf:
            #reduction de la variance du bruit
            bruit_std = max(1e-3, bruit_std / 2)
            
           
            
            
        else:
            #augmentation de la variance du bruit
            bruit_std = min(2, bruit_std * 2)
            
         

   

> <span style="color:green">- Préciser et justifier le critère d'arrêt que vous avez choisi.  </span>

> <span style="color:blue">Votre réponse: </span>

Compléter le code ci-dessous pour:
- rechercher une politique déterministe et une politique stochastique dans l'environnement CartPole
- tracer les performances pendant la recherche
- sauvegarder votre meilleure politique déterministe dans un fichier "best_pi_cartpole_det.npy" et votre meilleure politique stochastique dans un fichier "best_pi_cartpole_stoc.npy".
<span style="color:red"> Vous ajouterez ces 2 fichiers dans votre dépôt github ! </span> 

In [4]:
env = gym.make('CartPole-v1')
init_seed(5)

#TODO

    

## 1.4 Evaluation des politiques trouvées

Compléter la cellule ci-dessous pour évaluer les meilleures politiques déterministes et stochastiques que vous avez trouvées, et afficher leurs performances lors de l'évaluation (à partir d'état initiaux différents).


In [ ]:
env = gym.make('CartPole-v1')
init_seed(5)

meilleur_poids_det = np.load("best_pi_cartpole_det.npy")  # Charger les poids de la meilleure politique déterministe trouvée
meilleur_poids_stoc = np.load("best_pi_cartpole_stoc.npy")

#TODO


Afficher quelques épisodes avec vos meilleures politiques stochastiques et déterministes.

In [ ]:
#TODO


> <span style="color:green"> Analysez vos résultats.  </span>



> <span style="color:blue">Votre réponse: </span>

# 2 Environnement LunarLander avec actions continues

Vous devez maintenant appliquer la méthode par recherche directe de politique à l'environnement [LunarLander](https://gymnasium.farama.org/environments/box2d/lunar_lander/) **avec actions continues**.

> <span style="color:green"> Vous devez:  </span>
- 2.1 redéfinir une politique adaptée à un environnement avec actions continues   


> <span style="color:green">- Dans le cas d'une politique déterministe dans le LunarLander continu, qu'est-ce qui sera en sortie de la politique ?  </span>

> <span style="color:blue">Votre réponse: </span>

> <span style="color:green">- Dans le cas d'une politique stochastique dans le LunarLander continu, qu'est-ce qui sera en sortie de la politique ?  </span>

> <span style="color:blue">Votre réponse: </span>

In [ ]:
class PolitiqueAContinu():
    #TODO

- 2.2 effectuer une recherche dans l'espace des paramètres pour trouver une politique déterministe et une politique stochastique (réutiliser en la modifiant si nécessaire la méthode *recherche_directe* implémentée précédemment )


In [ ]:
#recherche de politique
env = gym.make("LunarLander-v3",continuous=True)
init_seed(5)

   #TODO

- 2.3  évaluer les meilleures politiques stochastique et déterministes trouvées

In [ ]:
#evaluation

   #TODO

> <span style="color:green"> Analysez vos résultats.  </span>